In [1]:
from pathlib import Path
import pandas as pd
import spatialdata as sd

import pickle
DATA_DIR = Path("/root/capsule/data")

/opt/conda/envs/xenium_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load data information

In [2]:
data_info = pickle.load(open('/root/capsule/code/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


In [3]:
cell_types_df = pd.DataFrame()
for i_mouse, mouse_id in enumerate(mouse_ids):
    Xenium_mapped_path = DATA_DIR / data_info[mouse_id]['xenium']['cell_types']
    sections = [int(s.name.split('_')[1]) for s in Xenium_mapped_path.glob("section_*") if s.is_dir()]
    sections.sort()
    
    for section in sections:
        df = pd.read_csv(Xenium_mapped_path / f'section_{section}/mapped_data/basic_results.csv', skiprows=4)
        df['mouse_id'] = mouse_id
        df['xenium_section'] = section
        df = df[(df['subclass_bootstrapping_probability']>.6)]
        cell_types_df = pd.concat([cell_types_df, df])
cell_types_df.rename(columns={'cell_id': 'cell_id - xenium'}, inplace=True)
cell_types_df.head()

,cell_id - xenium,class_label,class_name,class_bootstrapping_probability,subclass_label,subclass_name,subclass_bootstrapping_probability,supertype_label,supertype_name,supertype_bootstrapping_probability,cluster_label,cluster_name,cluster_alias,cluster_bootstrapping_probability,mouse_id,xenium_section
0,1,CS20230722_CLAS_33,33 Vascular,1.00,CS20230722_SUBC_330,330 VLMC NN,0.68,CS20230722_SUPT_1187,1187 VLMC NN_1,1.00,CS20230722_CLUS_5299,5299 VLMC NN_1,5266,0.50,778174,1
1,3,CS20230722_CLAS_33,33 Vascular,0.80,CS20230722_SUBC_333,333 Endo NN,0.99,CS20230722_SUPT_1193,1193 Endo NN_1,1.00,CS20230722_CLUS_5311,5311 Endo NN_1,5255,0.64,778174,1
2,8,CS20230722_CLAS_33,33 Vascular,0.99,CS20230722_SUBC_329,329 ABC NN,0.66,CS20230722_SUPT_1186,1186 ABC NN_1,1.00,CS20230722_CLUS_5293,5293 ABC NN_1,5256,0.51,778174,1
5,12,CS20230722_CLAS_33,33 Vascular,0.89,CS20230722_SUBC_329,329 ABC NN,0.75,CS20230722_SUPT_1186,1186 ABC NN_1,1.00,CS20230722_CLUS_5295,5295 ABC NN_1,5258,0.66,778174,1
6,13,CS20230722_CLAS_30,30 Astro-Epen,0.95,CS20230722_SUBC_319,319 Astro-TE NN,1.00,CS20230722_SUPT_1161,1161 Astro-TE NN_1,0.97,CS20230722_CLUS_5219,5219 Astro-TE NN_1,14930,0.82,778174,1


In [4]:
cell_types_df.to_csv('/root/capsule/code/tables/cell_types.csv', index=False)

In [4]:
cellxgene_df = []
for i_mouse, mouse_id in enumerate(mouse_ids):
    Xenium_processed_path = DATA_DIR / data_info[mouse_id]['xenium']['processed']
    sections = [int(s.name.split('_')[1].replace(".zarr","")) for s in Xenium_processed_path.glob("section_*") if s.is_dir()]
    sections.sort()
    
    for section in sections:
    
        section_path = Xenium_processed_path / f'section_{section}.zarr'
        sdata = sd.read_zarr(str(section_path))

        df = sdata['table'].to_df().reset_index().rename(columns={'index':'cell_id - xenium'})
        df.insert(0, 'xenium_section', [section]*len(df))
        df.insert(0, 'mouse_id', [mouse_id]*len(df))
        cellxgene_df.append(df)
cellxgene_df = pd.concat(cellxgene_df,axis=0)
cellxgene_df


,mouse_id,xenium_section,cell_id - xenium,2010300C02Rik,Acsbg1,Acta2,Acvrl1,Adamts2,Adamtsl1,Adcyap1,...,UnassignedCodeword_0471,UnassignedCodeword_0472,UnassignedCodeword_0473,UnassignedCodeword_0474,UnassignedCodeword_0475,UnassignedCodeword_0477,UnassignedCodeword_0478,UnassignedCodeword_0479,UnassignedCodeword_0480,UnassignedCodeword_0484
0,778174,1,1,1.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,778174,1,3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,778174,1,8,0.0,0.0,1.0,2.0,5.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,778174,1,10,0.0,0.0,0.0,6.0,8.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,778174,1,11,0.0,0.0,0.0,5.0,8.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18322,797371,24,18322,0.0,0.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18323,797371,24,18323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18324,797371,24,18324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18325,797371,24,18325,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
cellxgene_df.to_csv('/root/capsule/code/tables/cellxgene.csv', index=False)

In [5]:
ophys_xenium_match = pd.read_csv('/root/capsule/code/tables/ophys_xenium_match.csv')

In [6]:
main_subclasses = ['006 L4/5 IT CTX Glut',
'007 L2/3 IT CTX Glut',
'022 L5 ET CTX Glut',
'052 Pvalb Gaba',
'046 Vip Gaba',
'053 Sst Gaba',
'049 Lamp5 Gaba']

In [7]:
cell_types_coreg_df = ophys_xenium_match.merge(cell_types_df, on = ['mouse_id','xenium_section','cell_id - xenium'], how = 'left')
cell_types_coreg_df = cell_types_coreg_df[cell_types_coreg_df['subclass_name'].isin(main_subclasses)]
cell_types_coreg_df.to_csv('/root/capsule/code/tables/cell_types_coreg.csv',index=False)
cell_types_coreg_df

,unique_id,mouse_id,xenium_section,cell_id - zstack,cell_id - xenium,plane,x_loc,y_loc,z_loc,cell_id - session_1,...,subclass_label,subclass_name,subclass_bootstrapping_probability,supertype_label,supertype_name,supertype_bootstrapping_probability,cluster_label,cluster_name,cluster_alias,cluster_bootstrapping_probability
2,778174_4_470,778174,4,470,12715,VISp_0,273.799161,270.955895,100.833692,21,...,CS20230722_SUBC_006,006 L4/5 IT CTX Glut,0.65,CS20230722_SUPT_0028,0028 L4/5 IT CTX Glut_6,0.40,CS20230722_CLUS_0102,0102 L4/5 IT CTX Glut_6,200.0,0.49
6,778174_4_497,778174,4,497,12710,VISp_0,290.958952,290.312392,99.031798,28,...,CS20230722_SUBC_007,007 L2/3 IT CTX Glut,0.97,CS20230722_SUPT_0032,0032 L2/3 IT CTX Glut_4,0.58,CS20230722_CLUS_0118,0118 L2/3 IT CTX Glut_4,235.0,0.99
7,778174_4_513,778174,4,513,12721,VISp_0,295.963729,351.873050,100.762676,43,...,CS20230722_SUBC_046,046 Vip Gaba,0.91,CS20230722_SUPT_0178,0178 Vip Gaba_6,0.83,CS20230722_CLUS_0645,0645 Vip Gaba_6,626.0,0.51
10,778174_4_545,778174,4,545,12720,VISp_0,232.491496,348.890445,101.978989,41,...,CS20230722_SUBC_007,007 L2/3 IT CTX Glut,1.00,CS20230722_SUPT_0030,0030 L2/3 IT CTX Glut_2,0.95,CS20230722_CLUS_0109,0109 L2/3 IT CTX Glut_2,232.0,1.00
18,778174_5_516,778174,5,516,8285,VISp_0,443.221502,478.675855,101.669856,79,...,CS20230722_SUBC_007,007 L2/3 IT CTX Glut,0.99,CS20230722_SUPT_0030,0030 L2/3 IT CTX Glut_2,0.72,CS20230722_CLUS_0110,0110 L2/3 IT CTX Glut_2,233.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3539,797371_21_15149,797371,21,15149,8967,VISp_7,437.061133,472.606660,373.895914,206,...,CS20230722_SUBC_022,022 L5 ET CTX Glut,0.61,CS20230722_SUPT_0093,0093 L5 ET CTX Glut_4,0.64,CS20230722_CLUS_0372,0372 L5 ET CTX Glut_4,395.0,0.64
3544,797371_22_14977,797371,22,14977,7985,VISp_7,516.286434,634.210836,370.939941,298,...,CS20230722_SUBC_052,052 Pvalb Gaba,1.00,CS20230722_SUPT_0207,0207 Pvalb Gaba_3,0.99,CS20230722_CLUS_0740,0740 Pvalb Gaba_3,599.0,0.98
3545,797371_22_15069,797371,22,15069,10679,VISp_7,629.980086,488.764378,379.938797,223,...,CS20230722_SUBC_006,006 L4/5 IT CTX Glut,1.00,CS20230722_SUPT_0023,0023 L4/5 IT CTX Glut_1,1.00,CS20230722_CLUS_0073,0073 L4/5 IT CTX Glut_1,190.0,0.93
3546,797371_22_15076,797371,22,15076,7991,VISp_7,496.720414,602.781479,376.309866,279,...,CS20230722_SUBC_053,053 Sst Gaba,0.95,CS20230722_SUPT_0222,0222 Sst Gaba_9,0.64,CS20230722_CLUS_0801,0801 Sst Gaba_9,510.0,0.96


In [ ]:
cellxgene_coreg_df = ophys_xenium_match.merge(cellxgene_df, on = ['mouse_id','xenium_section','cell_id - xenium'], how = 'left')
cellxgene_coreg_df = cellxgene_coreg_df[cellxgene_coreg_df['unique_id'].isin(cell_types_coreg_df['unique_id'].values)]
cellxgene_coreg_df.to_csv('/root/capsule/code/tables/cellxgene_coreg.csv',index=False)
cellxgene_coreg_df

,unique_id,mouse_id,xenium_section,cell_id - zstack,cell_id - xenium,plane,x_loc,y_loc,z_loc,cell_id - session_1,...,UnassignedCodeword_0471,UnassignedCodeword_0472,UnassignedCodeword_0473,UnassignedCodeword_0474,UnassignedCodeword_0475,UnassignedCodeword_0477,UnassignedCodeword_0478,UnassignedCodeword_0479,UnassignedCodeword_0480,UnassignedCodeword_0484
2,778174_4_470,778174,4,470,12715,VISp_0,273.799161,270.955895,100.833692,21,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,778174_4_497,778174,4,497,12710,VISp_0,290.958952,290.312392,99.031798,28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,778174_4_513,778174,4,513,12721,VISp_0,295.963729,351.873050,100.762676,43,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,778174_4_545,778174,4,545,12720,VISp_0,232.491496,348.890445,101.978989,41,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18,778174_5_516,778174,5,516,8285,VISp_0,443.221502,478.675855,101.669856,79,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3539,797371_21_15149,797371,21,15149,8967,VISp_7,437.061133,472.606660,373.895914,206,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3544,797371_22_14977,797371,22,14977,7985,VISp_7,516.286434,634.210836,370.939941,298,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3545,797371_22_15069,797371,22,15069,10679,VISp_7,629.980086,488.764378,379.938797,223,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3546,797371_22_15076,797371,22,15076,7991,VISp_7,496.720414,602.781479,376.309866,279,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
